In [2]:
import os
import subprocess
import re
import statistics
import itertools

# CUDA code
CUDA_TEMPLATE = r'''
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define BM __BM__
#define BN __BN__
#define BK __BK__
#define TM __TM__
#define TN __TN__
#define NUM_THREADS ((BN / TN) * (BM / TM))

__global__ void CudaKernel(int n, const float *A, const float *B, float *C) {
    __shared__ float sA[BM][BK];
    __shared__ float sB[BK][BN];

    int bx = blockIdx.x; int by = blockIdx.y;
    int tx = threadIdx.x; int ty = threadIdx.y;

    int row = by * BM + ty * TM;
    int col = bx * BN + tx * TN;

    float res[TM][TN] = {0.0f};
    int tid = ty * (BN / TN) + tx;

    for (int p = 0; p < (n + BK - 1) / BK; ++p) {
        for (int i = 0; i < (BM * BK) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int a_row = idx / BK;
            int a_col = idx % BK;
            int g_a_row = by * BM + a_row;
            int g_a_col = p * BK + a_col;
            sA[a_row][a_col] = (g_a_row < n && g_a_col < n) ? A[g_a_row * n + g_a_col] : 0.0f;
        }

        for (int i = 0; i < (BK * BN) / NUM_THREADS; ++i) {
            int idx = i * NUM_THREADS + tid;
            int b_row = idx / BN;
            int b_col = idx % BN;
            int g_b_row = p * BK + b_row;
            int g_b_col = bx * BN + b_col;
            sB[b_row][b_col] = (g_b_row < n && g_b_col < n) ? B[g_b_row * n + g_b_col] : 0.0f;
        }
        __syncthreads();

        #pragma unroll
        for (int k = 0; k < BK; ++k) {
            float a_reg[TM];
            float b_reg[TN];
            for(int i=0; i<TM; i++) a_reg[i] = sA[ty * TM + i][k];
            for(int j=0; j<TN; j++) b_reg[j] = sB[k][tx * TN + j];
            for(int i=0; i<TM; i++) {
                for(int j=0; j<TN; j++) {
                    res[i][j] += a_reg[i] * b_reg[j];
                }
            }
        }
        __syncthreads();
    }

    for (int i = 0; i < TM; ++i) {
        for (int j = 0; j < TN; ++j) {
            int c_row = row + i;
            int c_col = col + j;
            if (c_row < n && c_col < n) {
                C[c_row * n + c_col] = res[i][j];
            }
        }
    }
}

int main(int argc, char **argv) {
    if (argc < 2) return 1;
    int n = atoi(argv[1]);
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes); float *h_b = (float*)malloc(bytes); float *h_c = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f; h_b[i * n + j] = 3.0f; h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes); cudaMalloc(&d_b, bytes); cudaMalloc(&d_c, bytes);
    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(BN / TN, BM / TM);
    dim3 blocksPerGrid((n + BN - 1) / BN, (n + BM - 1) / BM);

    cudaEvent_t start, stop;
    cudaEventCreate(&start); cudaEventCreate(&stop);

    cudaEventRecord(start);
    CudaKernel<<<blocksPerGrid, threadsPerBlock>>>(n, d_a, d_b, d_c);

    // --- HW Check ---
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("CRITICAL_ERROR: %s\n", cudaGetErrorString(err));
        cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
        free(h_a); free(h_b); free(h_c);
        return 1;
    }

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms = 0;
    cudaEventElapsedTime(&ms, start, stop);
    printf("Fast Execution Time: %.4f seconds\n", ms / 1000.0f);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);
    return 0;
}
'''

# Parameters to tune
BLOCK_SIZES = [32, 64, 128]
THREAD_SIZES = [4, 8]
BK_SIZES = [8, 16, 32]

# Settings
MATRIX_SIZE = 15000
NUM_RUNS = 3
CU_FILENAME = "temp_kernel.cu"
EXE_FILENAME = "./temp_run"

def check_constraints(BM, BN, BK, TM, TN):
    if BM % TM != 0 or BN % TN != 0: return False
    num_threads = (BM // TM) * (BN // TN)
    if num_threads > 1024: return False
    if (BM * BK) % num_threads != 0 or (BK * BN) % num_threads != 0: return False
    shared_mem_bytes = (BM * BK + BK * BN) * 4
    if shared_mem_bytes > 49152: return False
    return True

def run_benchmark():
    combinations = []

    for b_size in BLOCK_SIZES:
        for t_size in THREAD_SIZES:
            for bk in BK_SIZES:
                combinations.append({
                    'BM': b_size,
                    'BN': b_size,
                    'BK': bk,
                    'TM': t_size,
                    'TN': t_size
                })

    results = []

    print("=====================================================================")
    print(" Starting CUDA Hyperparameter Tuning...")
    print("=====================================================================")

    for config in combinations:
        if not check_constraints(**config):
            continue

        print(f"\n>>> Test Config: BM={config['BM']}, BN={config['BN']}, BK={config['BK']}, TM={config['TM']}, TN={config['TN']}")

        code = CUDA_TEMPLATE
        for key, val in config.items():
            code = code.replace(f"__{key}__", str(val))

        with open(CU_FILENAME, 'w') as f:
            f.write(code)

        compile_process = subprocess.run(["nvcc", "-arch=sm_75", "-O3", CU_FILENAME, "-o", EXE_FILENAME], capture_output=True)
        if compile_process.returncode != 0:
            print("  [!] Compilation Failed. Skipped.")
            continue

        times = []
        for i in range(1, NUM_RUNS + 1):
            run_process = subprocess.run([EXE_FILENAME, str(MATRIX_SIZE)], capture_output=True, text=True)

            # Check HW Error
            if "CRITICAL_ERROR" in run_process.stdout:
                err_msg = re.search(r"CRITICAL_ERROR:\s+(.*)", run_process.stdout).group(1)
                print(f"  [Run {i}/{NUM_RUNS}] Hardware Error: {err_msg}")
                break

            match = re.search(r"Fast Execution Time:\s+([0-9.]+)\s+seconds", run_process.stdout)
            if match:
                t = float(match.group(1))
                times.append(t)
                print(f"  [Run {i}/{NUM_RUNS}] Time: {t:.4f}s")
            else:
                print(f"  [Run {i}/{NUM_RUNS}] Execution Error Unknown!")

        if len(times) == NUM_RUNS:
            avg_time = statistics.mean(times)
            results.append((config, avg_time))
            print(f"---------------------------------------------------------------------")
            print(f"  Average Time: {avg_time:.4f}s")
            print(f"---------------------------------------------------------------------")
        else:
            print(f"---------------------------------------------------------------------")
            print(f"Configuration Discarded")
            print(f"---------------------------------------------------------------------")

    results.sort(key=lambda x: x[1])
    print("\n=====================================================================")
    print(" Results ")
    print("=====================================================================")
    for i, (config, avg_time) in enumerate(results):
        print(f"{i+1}. Time: {avg_time:.4f}s | Config: {config}")

    if os.path.exists(CU_FILENAME): os.remove(CU_FILENAME)
    if os.path.exists(EXE_FILENAME): os.remove(EXE_FILENAME)

if __name__ == "__main__":
    run_benchmark()

 Starting CUDA Hyperparameter Tuning...

>>> Test Config: BM=32, BN=32, BK=8, TM=4, TN=4
  [Run 1/3] Time: 3.5007s
  [Run 2/3] Time: 3.5418s
  [Run 3/3] Time: 3.6035s
---------------------------------------------------------------------
  Average Time: 3.5487s
---------------------------------------------------------------------

>>> Test Config: BM=32, BN=32, BK=16, TM=4, TN=4
  [Run 1/3] Time: 3.5711s
  [Run 2/3] Time: 3.5706s
  [Run 3/3] Time: 3.6189s
---------------------------------------------------------------------
  Average Time: 3.5869s
---------------------------------------------------------------------

>>> Test Config: BM=32, BN=32, BK=32, TM=4, TN=4
  [Run 1/3] Time: 3.4436s
  [Run 2/3] Time: 3.4917s
  [Run 3/3] Time: 3.5298s
---------------------------------------------------------------------
  Average Time: 3.4884s
---------------------------------------------------------------------

>>> Test Config: BM=32, BN=32, BK=8, TM=8, TN=8
  [Run 1/3] Time: 4.9695s
  [Run 2/3